## LAB | LangChain

In [ ]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

In [5]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

#### Checking Hawkins Records

In [13]:
@tool
def check_hawkins_records(query:str)->str:
    """Check the Hawkins National Laboratory records for information on a topic.
    Searches for the keywords: portal, monsters, psychics, electricity."""
    records = {
        "portal": "Hawkins Lab records: A gateway to another dimension was opened beneath the lab. Status: unstable, do not approach.",
        "monsters": "Hawkins Lab records: Multiple hostile entities have been catalogued. Classification: Demogorgon, Mind Flayer. Threat level: extreme.",
        "psychics": "Hawkins Lab records: Test subjects with telekinetic abilities were studied under Dr. Brenner. Subject Eleven is of particular interest.",
        "electricity": "Hawkins Lab records: Unexplained power surges and flickering lights are strongly correlated with activity from the other side.",
        }
    query_lower = query.lower()
    for key, record in records.items():
        if key in query_lower:
            return record
    return f"No record found for '{query}'. Try: portal, monsters, psychics, or electricity."


print(check_hawkins_records.invoke({"query": "What do we know about the Portal?"}))


Hawkins Lab records: A gateway to another dimension was opened beneath the lab. Status: unstable, do not approach.


In [15]:
@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Cast interdimensional spells to deal with a problem.
    creativity_level can be 'low', 'medium', or 'high' and controls how many spells are cast."""
    levels = {"low": 1, "medium": 2, "high": 3}
    n = levels.get(creativity_level.lower(), 2)

    spells = [
        f"Spell 1: Open a tiny portal and push '{problem}' through to the Upside Down.",
        f"Spell 2: Flicker the lights three times and whisper '{problem}' to the walls.",
        f"Spell 3: Summon Eleven's focus and telekinetically fling '{problem}' away.",
        f"Spell 4: Wrap '{problem}' in Christmas lights and ask Joyce to decode it.",
    ]

    return "\n".join(random.sample(spells, n))



print(cast_interdimensional_spell.invoke({"problem": "my wifi is down"}))
print(cast_interdimensional_spell.invoke({"problem": "my wifi is down", "creativity_level": "high"}))

Spell 2: Flicker the lights three times and whisper 'my wifi is down' to the walls.
Spell 3: Summon Eleven's focus and telekinetically fling 'my wifi is down' away.
Spell 3: Summon Eleven's focus and telekinetically fling 'my wifi is down' away.
Spell 4: Wrap 'my wifi is down' in Christmas lights and ask Joyce to decode it.
Spell 1: Open a tiny portal and push 'my wifi is down' through to the Upside Down.


In [16]:
@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the party for advice. Mention a party member by name (Mike, Dustin, Lucas, or Will)
    to hear their take; otherwise the whole party weighs in."""
    wisdom = {
        "mike": "Mike says: 'Stay calm, stick together, and make a plan. Nobody goes off alone.'",
        "dustin": "Dustin says: 'Okay, hear me out. We need science, snacks, and probably a compass.'",
        "lucas": "Lucas says: 'This is a bad idea. But fine, I'll bring the wrist rocket.'",
        "will": "Will says: 'I can sense something is off. Trust me, we should hurry.'",
    }
    question_lower = question.lower()
    for member, advice in wisdom.items():
        if member in question_lower:
            return advice
    return "The party huddles together, argues for a while, and concludes: 'We should probably ask Mike.'"

In [20]:
print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

The Demogorgon sniffs 'My lights keep flickering' and roars. It seems... hungry, not helpful.
Hawkins Lab records: A gateway to another dimension was opened beneath the lab. Status: unstable, do not approach.
Spell 3: Summon Eleven's focus and telekinetically fling 'Demodogs in my garden' away.
Spell 1: Open a tiny portal and push 'Demodogs in my garden' through to the Upside Down.
Spell 2: Flicker the lights three times and whisper 'Demodogs in my garden' to the walls.
The party huddles together, argues for a while, and concludes: 'We should probably ask Mike.'


In [21]:
tools=[consult_demogorgon,check_hawkins_records,cast_interdimensional_spell,gather_party_wisdom]

system_prompt="""
You are a Hawkins Investigator, an agent who helps people deal with strange problems.

Tool use:
- Be creative with your tools. Combine several of them when it helps, and try different ones if the first result isn't useful.
- You MUST call at least one tool before answering. Never respond without using a tool first.

Answering:
- Answer ONLY from the results the tools return. Do not add facts, advice, or lore from your own knowledge.
- If the tool results don't cover something, say so instead of making something up.

Format:
- End every answer with a line in this form: "Tools used: <tool_name>, <tool_name>
"""
agent=create_agent(model=model, tools=tools, system_prompt=system_prompt)

print([t.name for t in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


In [24]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

results = []
for complaint in complaints:
    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    results.append(result)
    print(f"Complaint: {complaint}")
    print(f"Answer: {result['messages'][-1].content}")
    print("-" * 60)

Complaint: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
Answer: The Hawkins National Laboratory records do not contain any information on flickering lights. However, the Demogorgon suggests that you "turn off the lights" in response to your situation with the Christmas lights and noises from the wall. 

Tools used: check_hawkins_records, consult_demogorgon
------------------------------------------------------------
Complaint: There is a weird portal in my basement. What's the plan?
Answer: The Hawkins Lab records indicate that a gateway to another dimension, like the one in your basement, is unstable and should not be approached. Mike suggests staying calm, sticking together, and making a plan. It's important that nobody goes off alone.

Tools used: check_hawkins_records, gather_party_wisdom
------------------------------------------------------------
Complaint: A Demodog ate my homework. I need a creative solution, creativity level high!
Answ

In [25]:
from collections import Counter

all_tools = []
for r in results:
    order = [m.name for m in r["messages"] if m.type == "tool"]
    all_tools.extend(order)
    print(" → ".join(order) if order else "(no tools called)")

print()
counts = Counter(all_tools)
for name, n in counts.most_common():
    print(f"{name}: {n}")

check_hawkins_records → consult_demogorgon
check_hawkins_records → gather_party_wisdom → gather_party_wisdom
cast_interdimensional_spell
consult_demogorgon → check_hawkins_records

check_hawkins_records: 3
consult_demogorgon: 2
gather_party_wisdom: 2
cast_interdimensional_spell: 1


In [26]:
for msg in results[0]["messages"]:
    if msg.type == "human":
        print(f"QUESTION: {msg.content}\n")
    elif msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"TOOL CALL: {tc['name']}({tc['args']})")
    elif msg.type == "tool":
        print(f"TOOL OUTPUT ({msg.name}): {msg.content}\n")

print("FINAL ANSWER:")
print(results[0]["messages"][-1].content)

QUESTION: My Christmas lights keep flickering and I hear noises from the wall. What is going on?

TOOL CALL: check_hawkins_records({'query': 'flickering lights'})
TOOL CALL: consult_demogorgon({'complaint': 'flickering Christmas lights and noises from the wall'})
TOOL OUTPUT (check_hawkins_records): No record found for 'flickering lights'. Try: portal, monsters, psychics, or electricity.

TOOL OUTPUT (consult_demogorgon): The Demogorgon tilts its petal-face at 'flickering Christmas lights and noises from the wall'. Interpreted as: 'turn off the lights'.

FINAL ANSWER:
The Hawkins National Laboratory records do not contain any information on flickering lights. However, the Demogorgon suggests that you "turn off the lights" in response to your situation with the Christmas lights and noises from the wall. 

Tools used: check_hawkins_records, consult_demogorgon


In [27]:
def structured_handler(complaint: str) -> str:
    records = check_hawkins_records.invoke({"query": complaint})
    wisdom = gather_party_wisdom.invoke({"question": complaint})
    spell = cast_interdimensional_spell.invoke(
        {"problem": complaint, "creativity_level": "low"}
    )
    return (
        f"RECORDS:\n{records}\n\n"
        f"PARTY WISDOM:\n{wisdom}\n\n"
        f"SPELL:\n{spell}"
    )

print(structured_handler(complaints[1]))

RECORDS:
Hawkins Lab records: A gateway to another dimension was opened beneath the lab. Status: unstable, do not approach.

PARTY WISDOM:
The party huddles together, argues for a while, and concludes: 'We should probably ask Mike.'

SPELL:
Spell 3: Summon Eleven's focus and telekinetically fling 'There is a weird portal in my basement. What's the plan?' away.


| | Agent | Stuctured | 
| --- | --- | --- | 
| Same tools every run? | No | Yes |
| Tool order predcitable | No | Yes|
| Can invent facts|  Yes | No|
| Easy Traced | No | Yes |
| Best for which client| Clients with open-ended, varied questions who want readable, adaptive answers |Clients who need reliable, auditable, repeatable results | 